*0.1 Python for GenAI*

# JSON schema generation

**The situation.** Your accounts team forwards supplier invoices as plain-text emails. You want the model to pull out the invoice number, the line items and the total, so the numbers can go straight into the accounting system. The first version asked the model: *"Extract the invoice as JSON."*

**What went wrong.** Nine times out of ten it worked. The tenth time the reply began with *"Sure! Here is the invoice:"* — or the field was called `Total` instead of `total`, or the amount came back as the text `"1,250.50"`. The parser crashed on 2% of emails, forever, and someone checked them by hand every morning.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**The fix: send the shape with the request.** A Pydantic model *is* a description of a shape. Pydantic can write that description out as a JSON schema — a formal list of fields and types. Send the schema with the request, and the provider only produces output that fits it. No sentence first, no renamed fields, no text where a number should be.

In [2]:
from pydantic import BaseModel, Field


class LineItem(BaseModel):
    description: str
    quantity: int = Field(ge=1)
    unit_price: float = Field(ge=0)


class Invoice(BaseModel):
    invoice_id: str = Field(pattern=r"^INV-\d{4}$", description="Format INV-0000")
    supplier: str
    items: list[LineItem]
    total: float = Field(description="Grand total as a number, no currency symbol")


schema = Invoice.model_json_schema()
print("required fields:", schema["required"])
print("nested types:", list(schema["$defs"]))
assert "items" in schema["required"]

required fields: ['invoice_id', 'supplier', 'items', 'total']
nested types: ['LineItem']


**Reading the output.** The schema lists the four required fields and the nested `LineItem` type — generated from the class, nothing written by hand.

**Now the email.** `client.beta.chat.completions.parse` sends the schema along with the text and hands back an `Invoice` object, already parsed and checked.

In [3]:
import json

from openai import OpenAI

email_text = """
Hi, please find our invoice below.
Invoice INV-0077 from Globex Supplies.
3 x cable set at 10.00 each, 1 x router at 25.00.
Total due: 55.00 USD. Thanks!
"""
client = OpenAI(timeout=30)
response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract the invoice from this email:\n{email_text}"}],
    response_format=Invoice,
    temperature=0,
)
invoice = response.choices[0].message.parsed
print(json.dumps(invoice.model_dump(), indent=2))
assert (
    invoice.invoice_id == "INV-0077" and len(invoice.items) == 2 and abs(invoice.total - 55) < 0.01
)

{
  "invoice_id": "INV-0077",
  "supplier": "Globex Supplies",
  "items": [
    {
      "description": "cable set",
      "quantity": 3,
      "unit_price": 10.0
    },
    {
      "description": "router",
      "quantity": 1,
      "unit_price": 25.0
    }
  ],
  "total": 55.0
}


**Reading the output.** An `Invoice` object with two line items and total 55.0 — a number, not `"55.00 USD"`. There is no parsing code anywhere: the shape was enforced by the provider and checked by Pydantic on the way in.

**The rule to remember.** Whenever the model's answer goes into code rather than to a person, send a schema. "Please answer in JSON" is a request; a schema is a guarantee.

| Use it when | Don't when | Instead use |
|---|---|---|
| model output feeds code: extraction, routing, tool arguments | the answer is for a person to read (a chat reply, a summary) | JSON mode when you only need valid JSON and no fixed shape |

**Watch out**
- Write a `description` on each field. The model reads them to decide what goes where.
- A required field the model cannot find in the text will be invented. Make such fields optional and check for `None`.
- The shape is guaranteed; the values are not. A `total` of 55.0 can still be wrong — keep your validators.